# B — Zero-shot BART-MNLI trên GoEmotions

Notebook hướng dẫn chạy phần B, đọc kết quả thật và hiểu cơ chế. Không tự fine-tune. Xem `docs/ZERO_SHOT.md` để hiểu protocol, smoke và resume.

## 1. Bài toán

Văn bản tiếng Anh → 28 cặp văn bản/giả thuyết → điểm NLI cho mỗi nhãn → ngưỡng → tập nhãn. Ví dụ giả thuyết: `This text expresses joy.`. Điểm này chưa phải xác suất cảm xúc được hiệu chuẩn. `multi_label=True` cho phép nhiều nhãn cùng cao.

B dùng `facebook/bart-large-mnli` đã học MNLI; nhóm không cập nhật trọng số trên GoEmotions. A học TF-IDF/LR từ train; C fine-tune 3 kiến trúc.

In [ ]:
from pathlib import Path
import json
import sys
import subprocess

ROOT = Path.cwd()
if not (ROOT / "src" / "zero_shot.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "src" / "zero_shot.py").exists(), "Hãy mở notebook từ repo"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
labels = json.loads((ROOT / "data/labels.json").read_text(encoding="utf-8"))
print("Số nhãn:", len(labels))
print("Repo:", ROOT)

## 2. Vì sao phải đổi thứ tự nhãn?

Pipeline xếp nhãn theo điểm; mỗi câu có thể xếp khác nhau. Ví dụ nhỏ dưới đây chỉ giải thích việc ghép cột, không phải đầu ra BART đã đo.

In [ ]:
from src.zero_shot import map_pipeline_scores

example = [{"labels": ["anger", "joy"], "scores": [0.9, 0.2]},
           {"labels": ["joy", "anger"], "scores": [0.8, 0.1]}]
print(map_pipeline_scores(example, ["joy", "anger"]))
# Cột 0 luôn là joy, cột 1 luôn là anger.

## 3. Chạy thử 8 câu

Cài môi trường theo `requirements-zero-shot.txt` trước. Lần đầu cần tải BART-large, CPU có thể lâu. Chọn `RUN_SMOKE = True` khi bạn sẵn sàng chạy thật; mặc định ô này không tải model.

Smoke chỉ kiểm code hoạt động; không dùng điểm smoke làm kết quả chính thức.

In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    subprocess.run([sys.executable, "-m", "scripts.run_zero_shot",
                    "--smoke", "--device", "cpu", "--batch-size", "2"],
                   cwd=ROOT, check=True)
else:
    print("Chưa chạy smoke. Đổi RUN_SMOKE=True để thực hiện.")

## 4. Chạy đủ validation

Từ terminal repo: `python -m scripts.run_zero_shot --device auto --batch-size 8`. Đây là 5.426 câu và 28 nhãn. Script chỉ đọc validation, không đọc train/test.

Từng batch được ghi trong `validation_checkpoints`. Chạy lại cùng lệnh để tiếp tục. Đổi revision, thiết bị, môi trường hay batch-size cần thí nghiệm riêng, không ghép lẫn checkpoint. `--limit` luôn là smoke.

Chỉ lấy số báo cáo khi metadata có `status=complete`, `smoke=false` và đủ 5.426 mẫu.

In [ ]:
output = ROOT / "data/processed/zero_shot/full"
metadata_path = output / "run_metadata.json"
if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    print("Trạng thái:", metadata["status"], "| Smoke:", metadata["smoke"])
    print("Số câu:", metadata["sample_count"])
    print("Model SHA:", metadata["model_revision"])
    for key in ["macro_f1", "micro_f1", "macro_precision", "macro_recall", "hamming_loss"]:
        print(f"{key}: {metadata['metrics'][key]:.4f}")
else:
    print("Chưa có kết quả full validation; không tạo điểm giả.")

## 5. Chọn ngưỡng và khóa trước test

Ngưỡng 0,5 là B mặc định. Nếu dùng nhãn validation để chọn ngưỡng global hoặc theo nhãn, ghi rõ **zero-shot về trọng số có hiệu chỉnh ngưỡng bằng validation**.

Dùng `python -m scripts.freeze_experiment --help` và `--run-dir data/processed/zero_shot/full` để tạo `final_protocol.json` (fixed/global/tuned). Sau khi chốt mode trên validation, chạy:

```powershell
python -m scripts.run_zero_shot --split test --device auto --batch-size 8 --protocol data/processed/zero_shot/full/final_protocol.json
```

CLI kiểm protocol và hash trước khi mở test. Điểm test dùng ngưỡng đã khóa; metadata test lưu riêng. Không chọn lại prompt, checkpoint hoặc ngưỡng dựa vào kết quả test.

## 6. Bàn giao và tự kiểm

- NPZ gồm ID, scores N×28 và label_names theo thứ tự chuẩn.
- Metadata chứa SHA checkpoint, data revision/hash, môi trường, thời gian và chỉ số shared.
- So sánh A/B/C phải cùng split, nhãn và metric; phân biệt validation/test.
- `python -m unittest tests.test_zero_shot -v` dùng dữ liệu giả kiểm mapping, resume, hash và protocol; không chứng minh chất lượng mô hình.

**Nguồn:** [model card](https://huggingface.co/facebook/bart-large-mnli), [HF pipeline](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines#transformers.ZeroShotClassificationPipeline), [GoEmotions ACL 2020](https://aclanthology.org/2020.acl-main.372/).